# 03. Point Cloud Segmentation Baseline

Run a classical DBSCAN segmentation baseline on the degraded virtual LiDAR scan. This is not expected to solve every occluded/touching fragment case; it is a transparent baseline for benchmark comparisons.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))

from segmentation.dbscan_baseline import segment_dbscan
from segmentation.metrics import clustering_scores

SCAN_DIR = ROOT / 'data' / 'scanned_pointclouds'
LABEL_DIR = ROOT / 'data' / 'labels'
FIGURE_DIR = ROOT / 'outputs' / 'figures'
TABLE_DIR = ROOT / 'outputs' / 'tables'
for path in [LABEL_DIR, FIGURE_DIR, TABLE_DIR]:
    path.mkdir(parents=True, exist_ok=True)

PARAMS = {
    'eps_m': 0.030,
    'min_samples': 8,
    'random_seed': 2026,
    'max_plot_points': 55_000,
}
scan = np.load(SCAN_DIR / 'synthetic_rockpile_exterior_only_scan.npz')
points = scan['points_xyz']
true_labels = scan['fragment_ids']
print(points.shape, len(np.unique(true_labels)))

In [ ]:
predicted_labels = segment_dbscan(points, eps_m=PARAMS['eps_m'], min_samples=PARAMS['min_samples'])
scores = clustering_scores(true_labels, predicted_labels)
scores.update({'eps_m': PARAMS['eps_m'], 'min_samples': PARAMS['min_samples'], 'n_points': len(points)})
scores_df = pd.DataFrame([scores])
scores_path = TABLE_DIR / 'segmentation_metrics.csv'
scores_df.to_csv(scores_path, index=False)

segmentation_path = LABEL_DIR / 'synthetic_rockpile_exterior_dbscan_segmentation.npz'
np.savez_compressed(segmentation_path, points_xyz=points, true_fragment_ids=true_labels, predicted_cluster_ids=predicted_labels)
scores_df

In [ ]:
rng = np.random.default_rng(PARAMS['random_seed'])
idx = rng.choice(len(points), size=min(PARAMS['max_plot_points'], len(points)), replace=False)

fig = plt.figure(figsize=(12, 5))
ax1 = fig.add_subplot(121, projection='3d')
ax1.scatter(points[idx, 0], points[idx, 1], points[idx, 2], c=true_labels[idx], s=1.0, cmap='tab20')
ax1.set_title('Ground-truth fragment IDs')
ax1.set_xlabel('x [m]'); ax1.set_ylabel('y [m]'); ax1.set_zlabel('z [m]')
ax1.view_init(elev=22, azim=-58)

ax2 = fig.add_subplot(122, projection='3d')
plot_labels = predicted_labels[idx].copy()
ax2.scatter(points[idx, 0], points[idx, 1], points[idx, 2], c=plot_labels, s=1.0, cmap='tab20')
ax2.set_title('DBSCAN predicted clusters')
ax2.set_xlabel('x [m]'); ax2.set_ylabel('y [m]'); ax2.set_zlabel('z [m]')
ax2.view_init(elev=22, azim=-58)
fig.suptitle(f"ARI={scores['adjusted_rand_index']:.3f}, clusters={scores['n_predicted_clusters']}")
fig.tight_layout()
figure_path = FIGURE_DIR / 'synthetic_rockpile_exterior_dbscan_segmentation.png'
fig.savefig(figure_path, dpi=180, bbox_inches='tight')
plt.show()
print(figure_path)